# RAG with LangChain

Before we dive into Verbatim RAG, let's first see how a traditional RAG system works. This will help us understand the problems that Verbatim RAG solves.

Traditional RAG systems:
1. **Split documents** into chunks
2. **Embed chunks** into vector space  
3. **Retrieve** relevant chunks for a query
4. **Generate answers** freely based on the retrieved context

The key issue: The LLM can generate plausible-sounding information that wasn't actually in the source documents!

In [1]:
# Install LangChain for comparison
!pip install "langchain==0.3.27" langchain-openai langchain-community faiss-cpu openai pypdf 

In [2]:
!pip install rich

In [3]:
from rich.console import Console

console = Console()

In [4]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI
from langchain.document_loaders import PyPDFLoader
from langchain.schema import Document as LangChainDocument


# Load PDFs using LangChain's PyPDFLoader (traditional PDF parsing)
console.print("Loading PDFs with PyPDFLoader...")
loader1 = PyPDFLoader("https://aclanthology.org/2025.bionlp-share.8.pdf")
loader2 = PyPDFLoader("https://aclanthology.org/2020.lrec-1.448.pdf")

langchain_docs = []
langchain_docs.extend(loader1.load())
langchain_docs.extend(loader2.load())

console.print(f"Loaded {len(langchain_docs)} document pages for traditional RAG")
console.print(f"\nSample content from PyPDFLoader:")
console.print(f"Page 1 preview: {langchain_docs[0].page_content[:200]}...")

Loading PDFs with PyPDFLoader...

Loaded 11 document pages for traditional RAG

Sample content from PyPDFLoader:

Page 1 preview: BioNLP 2025 Shared Tasks, pages 69–74
August 1, 2025 ©2025 Association for Computational Linguistics
KR Labs at ArchEHR-QA 2025: A Verbatim Approach for Evidence-Based
Question Answering
Ádám Kovács1,...

In [5]:
# Traditional RAG: Sentence-based text splitting for fairer comparison
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],  # tries to split at sentence boundaries first
    chunk_size=1024,
    chunk_overlap=50
)

# Split documents into chunks
splits = text_splitter.split_documents(langchain_docs)
console.print(f"Created {len(splits)} sentence-based chunks")

# Show a sample chunk
console.print("\nSample chunk:")
console.print(f"Content: {splits[0].page_content}...")
console.print(f"Metadata: {splits[0].metadata}")

Created 55 sentence-based chunks

Sample chunk:

Content: BioNLP 2025 Shared Tasks, pages 69–74
August 1, 2025 ©2025 Association for Computational Linguistics
KR Labs at ArchEHR-QA 2025: A Verbatim Approach for Evidence-Based
Question Answering
Ádám Kovács1, Paul Schmitt2, Gábor Recski1,2
1KR Labs
lastname@krlabs.eu
2TU Wien
firstname.lastname@tuwien.ac.at
Abstract
We present a lightweight, domain-agnostic ver-
batim pipeline for evidence -grounded ques-
tion answering. Our pipeline operates in two
steps: first, a sentence-level extractor flags
relevant note sentences using either zero-shot
LLM prompts or supervised ModernBERT
classifiers. Next, an LLM drafts a question-
specific template, which is filled verbatim
with sentences from the extraction step. This
prevents hallucinations and ensures traceabil-
ity. In the ArchEHR -QA 2025 shared task,
our system scored 42.01%, ranking top-10 in
core metrics and outperforming the organiser’s
70B-parameter Llama-3.3 baseline. We pub-
licly release our code and inference scripts un-
der an MIT license.
1 Introduction...

Metadata: {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': 
'2025-07-09T14:47:02+08:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': 
'2025-07-09T14:47:02+08:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 
3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'source': 
'https://aclanthology.org/2025.bionlp-share.8.pdf', 'total_pages': 6, 'page': 0, 'page_label': '69'}

In [8]:
import os
# Add your key here
# os.environ["OPENAI_API_KEY"] = 

In [9]:
# Set up OpenAI API (you need to have OPENAI_API_KEY set)
# Make sure you have: export OPENAI_API_KEY=your_api_key_here

from langchain.chat_models import ChatOpenAI

# Create embeddings and vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(splits, embeddings)

# Create the traditional QA chain with gpt-5.1
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(temperature=1.0, model_name="gpt-5.1"),
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    return_source_documents=True,
    verbose=False,
)

console.print("Traditional RAG system set up successfully!")


/tmp/ipykernel_83943/1203474703.py:12: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm=ChatOpenAI(temperature=1.0, model_name="gpt-5.1"),


Traditional RAG system set up successfully!

In [10]:
# Query the traditional RAG system
question = "How much synthetic data they generated in ArchEHR-QA 2025?"

result = qa_chain({"query": question})

console.print("## Traditional RAG Answer:")
console.print(result["result"])

console.print(f"\n## Source Documents Used ({len(result['source_documents'])}):")
for i, doc in enumerate(result["source_documents"][:3]):  # Show first 3 sources
    console.print(f"\n**Source {i + 1}:** {doc.metadata.get('title', 'Unknown')}")
    console.print(f"Content preview: {doc.page_content[:150]}...")

/tmp/ipykernel_83943/1675998636.py:4: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"query": question})


## Traditional RAG Answer:

They generated 3,915 synthetic clinical notes, which were then expanded into a sentence-level dataset of about 
58,000 synthetic training examples (each sentence labeled for relevance).

## Source Documents Used (5):

**Source 1:**

Content preview: explicitly marked with [START] and [END] tokens.
The full input was structured using the standard
BERT classification format. During fine-tuning,
we m...

**Source 2:**

Content preview: Sarvesh Soni and Dina Demner-Fushman. 2025a. A
dataset for addressing patient’s information needs
related to clinical course of hospitalization. arXiv...

**Source 3:**

Content preview: cally creating answer templates filled exclu-
sively with verbatim sentences selected from
the extraction phase.
We participated in the ArchEHR-QA 202...

## Problems with Traditional RAG

Traditional RAG systems like the one above have several issues:

### 1. **Hallucination Risk**
- The LLM can generate plausible-sounding but incorrect information
- Numbers might be rounded or approximated ("around 58,000" vs exact "58k")
- The model might combine information from multiple sources incorrectly

### 2. **Poor Traceability**  
- Hard to verify exactly where specific claims come from
- Source documents are provided but without precise mappings to claims
- Difficult to fact-check individual statements

### 3. **No Guarantee of Grounding**
- Even with source documents, there's no guarantee the answer only uses information from them
- The LLM might fill in gaps with its training knowledge

Let's try a different approach, Verbatim RAG!

# Verbatim RAG

<p align="center">
  <img src="https://github.com/KRLabsOrg/verbatim-rag/blob/main/assets/chiliground.png?raw=true" alt="ChiliGround Logo" width="400"/>
  <br><em>Chill, I Ground! 🌶 ️</em>
</p>

A minimalistic approach to Retrieval-Augmented Generation (RAG) that prevents hallucination by ensuring all generated content is explicitly derived from source documents.

[![PyPI](https://img.shields.io/pypi/v/verbatim-rag)](https://pypi.org/project/verbatim-rag/)
[![License](https://img.shields.io/badge/License-MIT-blue.svg)](https://opensource.org/licenses/MIT)
[![ACL 2025](https://img.shields.io/badge/ACL%20Anthology-2025.bionlp--share.8-blue)](https://aclanthology.org/2025.bionlp-share.8/)

## Concept

Traditional RAG systems retrieve relevant documents and then allow an LLM to freely generate responses based on that context. This can lead to hallucinations where the model invents facts not present in the source material.

Verbatim RAG solves this by extracting verbatim text spans from documents and composing responses entirely from these exact passages, with direct citations linking back to sources.

For extraction, we can use LLM-based span extractors or fine-tuned encoder-based models like ModernBERT. We've trained our own ModernBERT model for this purpose, which is available on [HuggingFace](https://huggingface.co/KRLabsOrg/verbatim-rag-modern-bert-v1) (we've trained it on the [RAGBench](https://huggingface.co/datasets/galileo-ai/ragbench) dataset).

With this approach, **the whole RAG pipeline can be run without any usage of LLMs**, and with using SPLADE embeddings, the pipeline can be run entirely on CPU, making it lightweight and efficient.


Lets try it out with a few papers and simple examples!

## First steps

In [11]:
# Install dependencies
!pip install verbatim-rag

In [12]:
# Define the schema
from verbatim_rag import VerbatimRAG, VerbatimIndex
from verbatim_rag.schema import DocumentSchema

# Add two papers
paper = DocumentSchema.from_url(
    url="https://aclanthology.org/2025.bionlp-share.8.pdf",
    title="KR Labs at ArchEHR-QA 2025: A Verbatim Approach for Evidence-Based Question Answering",
    doc_type="academic_paper",
    authors=["Adam Kovacs", "Paul Schmitt", "Gabor Recski"],
    conference="BioNLP",
    year=2025,
    category="nlp",
)

paper2 = DocumentSchema.from_url(
    url="https://aclanthology.org/2020.lrec-1.448.pdf",
    title="Better Together: Modern Methods Plus Traditional Thinking in NP Alignment",
    doc_type="academic_paper",
    authors=["Adam Kovacs", "Judit Acs", "Andras Kornai", "Gabor Recski"],
    conference="LREC",
    year=2020,
    category="nlp",
)

2025-12-05 07:51:12,450 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-05 07:51:12,530 - INFO - Going to convert document batch...
2025-12-05 07:51:12,531 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e15bc6f248154cc62f8db15ef18a8ab7
2025-12-05 07:51:12,545 - INFO - Loading plugin 'docling_defaults'
2025-12-05 07:51:12,548 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-12-05 07:51:12,561 - INFO - Loading plugin 'docling_defaults'
2025-12-05 07:51:12,565 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-12-05 07:51:12,565 - INFO - rapidocr cannot be used because onnxruntime is not installed.
2025-12-05 07:51:12,566 - INFO - easyocr cannot be used because it is not installed.
2025-12-05 07:51:13,174 - INFO - Accelerator device: 'cpu'
[INFO] 2025-12-05 07:51:13,195 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2025-12-05 07:51:13,243 [RapidOCR] download_file.py:60: Fil

In [13]:
# Papers are parsed into markdown
console.print(paper.content[:500])

## KR Labs at ArchEHR-QA 2025: A Verbatim Approach for Evidence-Based Question Answering

Ádám Kovács 1 , Paul Schmitt 2 , Gábor Recski 1 , 2

1 KR Labs lastname@krlabs.eu

2 TU Wien firstname.lastname@tuwien.ac.at

## Abstract

We present a lightweight, domain-agnostic verbatim pipeline for evidence-grounded question answering. Our pipeline 
operates in two steps: first, a sentence-level extractor flags relevant note sentences using either zero-shot LLM 
prompts or supervised ModernBERT classifie

In [14]:
# If you want, you can define your content manually
paper_manual = DocumentSchema(
    content="""
    # Title
    ## Authors
    Adam Kovacs, Paul Schmitt, Gabor Recski
    """,
    title="KR Labs at ArchEHR-QA 2025: A Verbatim Approach for Evidence-Based Question Answering",
    doc_type="academic_paper",
    authors=["Adam Kovacs", "Paul Schmitt", "Gabor Recski"],
    conference="BioNLP",
    year=2025,
    category="nlp",
)

## Chunking

Chunking is the process of splitting the content of a document into smaller, more manageable chunks. This is important because it allows us to retrieve the most relevant information from the document.

While in theory, chunking is a simple process, in practice it can be quite complex, but its always worth it to do it right.

In [15]:
from verbatim_rag.chunker_providers import MarkdownChunkerProvider


chunker = MarkdownChunkerProvider(
     min_chunk_size=500,
     max_chunk_size=5000,
)

chunks = chunker.chunk(paper.content)

console.print(f"Chunk 1: {chunks[1][0]}")
console.print(f"Chunk 2: {chunks[2][0]}")

Chunk 1: ## 1 Introduction

Modern question-answering (QA) and retrievalaugmented generation (RAG) systems play a vital role in many 
high-stakes domains for information extraction and generation tasks. In medicine, a typical use case involves 
clinicians asking questions based on a patient's electronic health record (EHR) notes, rather than manually sifting
through lengthy notes, which can be time-consuming. However, in practice, RAG and QA pipelines often misalign 
evidence and produce incorrect information, commonly referred to as hallucinations (Ji et al., 2023; Madsen et al.,
2024). We argue that a reliable QA system should guarantee complete traceability of answers. To tackle this 
problem, we propose a verbatim pipeline that clearly separates extraction and generation to mitigate 
hallucinations:

- Sentence-level extraction , using either zeroshot LLMs or supervised ModernBERT classifiers.
- Template-constrained generation , dynamically creating answer templates filled exclu-

sively with verbatim sentences selected from the extraction phase.

We participated in the ArchEHR-QA 2025 shared task on grounded question answering (QA) from electronic health 
records (EHRs). Our approach involved (i) utilizing a zero-shot gemma-3-27b-it 1 LLM (Team et al., 2025) and (ii) 
generating synthetic data for sentence extraction from EHRs to train a compact extractor. For this purpose, we 
trained a Clinical ModernBERT classifier (Lee et al., 2025; Warner et al., 2024), achieving performance comparable 
to the LLM extractor. Both extractors were then fed into the same LLM template generator. Our solution achieved an 
overall score of 42.01% , ranking in the top 10 for core metrics, and surpassed the organizers' 70Bparameter 
Llama-3.3 baseline by a large margin.

Our contributions include a modular, traceable QA architecture that mitigates hallucinations, a method to generate 
synthetic EHR question-answer corpus and train custom models. Additionally, we are releasing all the code on GitHub
2 under the MIT License. The remainder of the paper discusses background (Section 2), method (Section 3), and 
evaluation (Section 4).

Chunk 2: ## 2 Background

## 2.1 Dataset

Early clinical QA datasets such as emrQA (Pampari et al., 2018) and CliCR (Šuster and Daelemans, 2018) used 
fill-in-the-blank methods and lacked explicit sentence-level evidence. ArchEHR-QA (Soni and Demner-Fushman, 
2025b,a) addresses this by pairing clinician-authored questions with deidentified MIMIC-III (Johnson et al., 2016) 
notes, annotated at the sentence-level as essential , supplementary , or irrelevant . Answers must be concise

1 https://huggingface.co/google/gemma-3-27b-it 2 https://github.com/KRLabsOrg/verbatim-rag/ tree/archehr

(under 75 words) and explicitly cite relevant sentences.

Many document types (e.g. markdown) have a certain structure, that is something we shouldn't lose with the chunking process.

In [16]:
console.print(f"Chunk 1: {chunks[1][1]}")
console.print(f"Chunk 2: {chunks[2][1]}")

Chunk 1: ## 1 Introduction

Modern question-answering (QA) and retrievalaugmented generation (RAG) systems play a vital role in many 
high-stakes domains for information extraction and generation tasks. In medicine, a typical use case involves 
clinicians asking questions based on a patient's electronic health record (EHR) notes, rather than manually sifting
through lengthy notes, which can be time-consuming. However, in practice, RAG and QA pipelines often misalign 
evidence and produce incorrect information, commonly referred to as hallucinations (Ji et al., 2023; Madsen et al.,
2024). We argue that a reliable QA system should guarantee complete traceability of answers. To tackle this 
problem, we propose a verbatim pipeline that clearly separates extraction and generation to mitigate 
hallucinations:

- Sentence-level extraction , using either zeroshot LLMs or supervised ModernBERT classifiers.
- Template-constrained generation , dynamically creating answer templates filled exclu-

sively with verbatim sentences selected from the extraction phase.

We participated in the ArchEHR-QA 2025 shared task on grounded question answering (QA) from electronic health 
records (EHRs). Our approach involved (i) utilizing a zero-shot gemma-3-27b-it 1 LLM (Team et al., 2025) and (ii) 
generating synthetic data for sentence extraction from EHRs to train a compact extractor. For this purpose, we 
trained a Clinical ModernBERT classifier (Lee et al., 2025; Warner et al., 2024), achieving performance comparable 
to the LLM extractor. Both extractors were then fed into the same LLM template generator. Our solution achieved an 
overall score of 42.01% , ranking in the top 10 for core metrics, and surpassed the organizers' 70Bparameter 
Llama-3.3 baseline by a large margin.

Our contributions include a modular, traceable QA architecture that mitigates hallucinations, a method to generate 
synthetic EHR question-answer corpus and train custom models. Additionally, we are releasing all the code on GitHub
2 under the MIT License. The remainder of the paper discusses background (Section 2), method (Section 3), and 
evaluation (Section 4).

Chunk 2: ## 2 Background

## 2.1 Dataset

Early clinical QA datasets such as emrQA (Pampari et al., 2018) and CliCR (Šuster and Daelemans, 2018) used 
fill-in-the-blank methods and lacked explicit sentence-level evidence. ArchEHR-QA (Soni and Demner-Fushman, 
2025b,a) addresses this by pairing clinician-authored questions with deidentified MIMIC-III (Johnson et al., 2016) 
notes, annotated at the sentence-level as essential , supplementary , or irrelevant . Answers must be concise

1 https://huggingface.co/google/gemma-3-27b-it 2 https://github.com/KRLabsOrg/verbatim-rag/ tree/archehr

(under 75 words) and explicitly cite relevant sentences.

## Indexing

We can now index the chunks. This is the process of converting the chunks into a vector space, so we can use them for retrieval. Usually very resource intensive part if you have a lot of documents.

In [17]:
from verbatim_rag.embedding_providers import SentenceTransformersProvider
from verbatim_rag.vector_stores import LocalMilvusStore

dense_provider = SentenceTransformersProvider(
    model_name="ibm-granite/granite-embedding-english-r2", device='cpu'
)
vector_store = LocalMilvusStore(
    db_path="./rag_lecture.db",
    collection_name='rag_lecture',
    dense_dim=dense_provider.get_dimension(),
    enable_dense=True,
    enable_sparse=False,
    nlist=16384,
)
index = VerbatimIndex(
        vector_store=vector_store,
        dense_provider=dense_provider,
        chunker_provider=chunker,
    )

index.add_documents([paper, paper2])

2025-12-05 07:52:14,228 - INFO - PyTorch version 2.9.1 available.
2025-12-05 07:52:14,745 - INFO - Load pretrained SentenceTransformer: ibm-granite/granite-embedding-english-r2
2025-12-05 07:52:19,318 - INFO - Loaded SentenceTransformers model: ibm-granite/granite-embedding-english-r2
/home/recski/miniconda3/envs/nlp_course/lib/python3.12/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
2025-12-05 07:52:20,145 - INFO - Connected to Milvus Lite: ./rag_lecture.db
Adding documents:   0%|                                                                                                                                                                                    | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-12-05 07:53:16,102 - INFO - Added 15 vectors to Milvus
2025-12-05 07:53:16,126 - INFO - Added 1 documents to Milvus
Adding documents:  50%|██████████████████████████████████████████████████████████████████████████████████████                                                                                      | 1/2 [00:55<00:55, 55.97s/it]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-12-05 07:54:00,724 - INFO - Added 12 vectors to Milvus
2025-12-05 07:54:00,739 - INFO - Added 1 documents to Milvus
Adding documents: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [01:40<00:00, 50.29s/it]


In [18]:
chunks = index.get_all_chunks()

# First chunk
console.print(f"Chunk 1: {chunks[0]}")

Chunk 1: SearchResult(id='0543ff9e-0d81-4da7-bce5-e6d04b27e2cc', score=1.0, metadata={'document_id': 
'3d52a019-5a15-4258-ab58-0d8ae3dcda97', 'title': 'KR Labs at ArchEHR-QA 2025: A Verbatim Approach for 
Evidence-Based Question Answering', 'source': 'https://aclanthology.org/2025.bionlp-share.8.pdf', 'doc_type': 
'academic_paper', 'content_type': 'pdf', 'chunk_type': 'paragraph', 'chunk_number': 4, 'page_number': 0, 
'created_at': '2025-12-03T20:06:11.707053', 'authors': ['Adam Kovacs', 'Paul Schmitt', 'Gabor Recski'], 
'conference': 'BioNLP', 'year': 2025, 'category': 'nlp'}, text='## 2.3 Synthetic Training Data\n\nDue to limited 
access and annotation restrictions, obtaining sentence-level labeled clinical datasets is challenging. Recent works
address this by generating synthetic data via perturbation or LLM prompting (Niu et al., 2024; Lozano et al., 2023;
Frayling et al., 2024; Bai et al., 2024). We follow this approach, generating synthetic EHR snippets, 
clinician-style questions, and sentence relevance annotations (details in Section 3.3).\n\n## 3 Method\n\n## 3.1 
System Overview\n\nFigure 1 depicts our system architecture. First, an extraction step identifies relevant 
sentences from the input (patient narrative, clinician question, and note excerpt). We implemented both zero-shot 
and supervised models. Second, the generation step uses gemma-3-27b-it to dynamically draft an answer template, 
filled verbatim with extracted sentences. If exceeding 75 words, answers are compressed via a summarization prompt,
preserving sentence-level citations.\n\n', enhanced_text="## 2.3 Synthetic Training Data\n\nDue to limited access 
and annotation restrictions, obtaining sentence-level labeled clinical datasets is challenging. Recent works 
address this by generating synthetic data via perturbation or LLM prompting (Niu et al., 2024; Lozano et al., 2023;
Frayling et al., 2024; Bai et al., 2024). We follow this approach, generating synthetic EHR snippets, 
clinician-style questions, and sentence relevance annotations (details in Section 3.3).\n\n## 3 Method\n\n## 3.1 
System Overview\n\nFigure 1 depicts our system architecture. First, an extraction step identifies relevant 
sentences from the input (patient narrative, clinician question, and note excerpt). We implemented both zero-shot 
and supervised models. Second, the generation step uses gemma-3-27b-it to dynamically draft an answer template, 
filled verbatim with extracted sentences. If exceeding 75 words, answers are compressed via a summarization prompt,
preserving sentence-level citations.\n\n\n\n---\nDocument: KR Labs at ArchEHR-QA 2025: A Verbatim Approach for 
Evidence-Based Question Answering\nSource: https://aclanthology.org/2025.bionlp-share.8.pdf\nDoc Type: 
academic_paper\nContent Type: DocumentType.PDF\nCreated At: 2025-12-03T20:06:11.707053\nAuthors: ['Adam Kovacs', 
'Paul Schmitt', 'Gabor Recski']\nConference: BioNLP\nYear: 2025\nCategory: nlp")

In [19]:
# You can filter by metadata, very useful for document type, conference, year, user_id, etc.
index.query(
    filter="metadata['title'] == 'KR Labs at ArchEHR-QA 2025: A Verbatim Approach for Evidence-Based Question Answering'",
    k=1,
)

[SearchResult(id='0543ff9e-0d81-4da7-bce5-e6d04b27e2cc', score=1.0, metadata={'document_id': '3d52a019-5a15-4258-ab58-0d8ae3dcda97', 'title': 'KR Labs at ArchEHR-QA 2025: A Verbatim Approach for Evidence-Based Question Answering', 'source': 'https://aclanthology.org/2025.bionlp-share.8.pdf', 'doc_type': 'academic_paper', 'content_type': 'pdf', 'chunk_type': 'paragraph', 'chunk_number': 4, 'page_number': 0, 'created_at': '2025-12-03T20:06:11.707053', 'authors': ['Adam Kovacs', 'Paul Schmitt', 'Gabor Recski'], 'conference': 'BioNLP', 'year': 2025, 'category': 'nlp'}, text='## 2.3 Synthetic Training Data\n\nDue to limited access and annotation restrictions, obtaining sentence-level labeled clinical datasets is challenging. Recent works address this by generating synthetic data via perturbation or LLM prompting (Niu et al., 2024; Lozano et al., 2023; Frayling et al., 2024; Bai et al., 2024). We follow this approach, generating synthetic EHR snippets, clinician-style questions, and sentence

## Lets do some RAG

Now that we have our index, we can use it to answer questions.

We'll use the `VerbatimRAG` class to answer questions.

In [20]:
from verbatim_rag.core import LLMClient

llm_client = LLMClient(model="gpt-5.1", temperature=1.0)

rag = VerbatimRAG(index, llm_client=llm_client)

response = rag.query("How much synthetic data they generated in Kovacs et al. 2025?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Extracting relevant spans...
Extracting spans (batch mode)...


2025-12-05 07:54:06,460 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Processing spans...
Generating response...


2025-12-05 07:54:09,227 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [21]:
console.print(response.answer)

Kovacs et al. (2025) report the amount and composition of synthetic data as follows:

- **Total synthetic notes generated:** [1] This approach yielded 3915 synthetic notes. 
- **Construction of the synthetic corpus:** [2] Ultimately, selecting each sentence with their relevance from the 
note excerpts, we constructed a comprehensive dataset of 58k synthetic training examples, each labeled at the 
sentence level, which formed the training set for our Clinical ModernBERT classifier.

In [22]:
console.print(response.structured_answer)

StructuredAnswer(
    text='Kovacs et al. (2025) report the amount and composition of synthetic data as follows:\n\n- **Total 
synthetic notes generated:** [1] This approach yielded 3915 synthetic notes. \n- **Construction of the synthetic 
corpus:** [2] Ultimately, selecting each sentence with their relevance from the note excerpts, we constructed a 
comprehensive dataset of 58k synthetic training examples, each labeled at the sentence level, which formed the 
training set for our Clinical ModernBERT classifier.',
    citations=[
        Citation(
            text='This approach yielded 3915 synthetic notes.',
            doc_index=2,
            highlight_index=0,
            number=1,
            type='display'
        ),
        Citation(
            text='Ultimately, selecting each sentence with their relevance from the note excerpts, we constructed a
comprehensive dataset of 58k synthetic training examples, each labeled at the sentence level, which formed the 
training set for our Clinical ModernBERT classifier.',
            doc_index=2,
            highlight_index=1,
            number=2,
            type='display'
        ),
        Citation(
            text='This approach yielded 3915 synthetic notes.',
            doc_index=3,
            highlight_index=0,
            number=3,
            type='reference'
        ),
        Citation(
            text='Ultimately, selecting each sentence with their relevance from the note excerpts, we constructed a
comprehensive dataset of 58k synthetic training examples, each labeled at the sentence level, which formed the 
training set for our Clinical ModernBERT classifier.',
            doc_index=3,
            highlight_index=1,
            number=4,
            type='reference'
        )
    ]
)

By default, **VerbatimRAG** uses a dynamic template generation strategy, generating a template for each question with LLMs to ensure the best possible answer.

A special mode is also available, where you want full control over your answer, that case you can use the `static` mode, where you can provide the exact template you want your answer to be in.

Try it out:

In [23]:
template = """
Thanks for your question!

You can find the answer to your question in the following sources:
[RELEVANT_SENTENCES]
"""


rag.template_manager.use_static_mode(template)

response = rag.query("How much synthetic data they generated in Kovacs et al. 2025?")

console.print(response.answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Extracting relevant spans...
Extracting spans (batch mode)...


2025-12-05 07:54:14,143 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Processing spans...
Generating response...


Thanks for your question!

You can find the answer to your question in the following sources:
[1] Wegenerated synthetic data via few-shot prompting with gemma-3-27b-it . Each prompt provided dynamic examples 
from the development set to ensure diversity. The LLM generated synthetic instances comprising de-identified 
clinical note excerpts, patient narratives, clinician-authored questions, and binary relevance labels. This 
approach yielded 3915 synthetic notes. We varied the few-shot examples across multiple runs, as static prompting 
resulted in repetitive outputs. This variation greatly increased lexical and semantic diversity, aligning with 
other work in synthetic data generation (Li et al., 2023; Tang et al., 2023; Xu et al., 2024). Ultimately, 
selecting each sentence with their relevance from the note excerpts, we constructed a comprehensive dataset of 58k 
synthetic training examples, each labeled at the sentence level, which formed the training set for our Clinical 
ModernBERT classifier.